# FieldMind Africa — 02 QLoRA train and export

Use a freely allocated NVIDIA GPU in Kaggle or Colab. This notebook uses no paid API, GPU credit, or experiment tracker. GPU availability is not guaranteed by either service. Save artifacts before the session ends.

In [ ]:
REPO_URL = "https://github.com/OtienoKeith/fieldmind-africa.git"
REPO_DIR = "fieldmind-africa"
from pathlib import Path
import os, subprocess
base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content') if Path('/content').exists() else Path.cwd()
repo = base / REPO_DIR
if not (repo / 'scripts/train_qlora.py').exists():
    if not REPO_URL:
        raise RuntimeError('Set REPO_URL or upload/clone the repository first.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo)], check=True)
os.chdir(repo)
print('Repository:', repo)

In [ ]:
# Unsloth must be installed before importing transformers/trl. Restart the runtime if pip asks.
%pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo datasets

In [ ]:
import torch, json
from pathlib import Path
assert torch.cuda.is_available(), 'Enable a free GPU accelerator before training.'
print(torch.cuda.get_device_name(0))
if not Path('data/processed/manifest.json').exists():
    subprocess.run(['python', 'scripts/prepare_dataset.py'], check=True)
manifest = json.loads(Path('data/processed/manifest.json').read_text())
assert manifest['mode'] == 'open_farmerchat', 'Run the real open-data build, not seed-only mode.'
print(manifest['counts'])

In [ ]:
!python scripts/validate_data.py
!python scripts/train_qlora.py --config config/training.json --train data/processed/train.jsonl --validation data/processed/validation.jsonl

## Sanity check the merged model
This local generation is not a benchmark. It catches broken merges before spending time on GGUF conversion.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
merged = Path('artifacts/fieldmind-merged-16bit')
tok = AutoTokenizer.from_pretrained(merged)
m = AutoModelForCausalLM.from_pretrained(merged, torch_dtype='auto', device_map='auto')
messages = [{'role':'user','content':'My cassava leaves are curling after heavy rain. Should I buy fungicide now?'}]
inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(m.device)
out = m.generate(inputs, max_new_tokens=300, do_sample=False)
print(tok.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))
del m
torch.cuda.empty_cache()

## Build llama.cpp and create the quantization tournament
This stage can run on the notebook CPU after training. It creates F16, Q3_K_M, Q4_K_M, Q5_K_M, and Q6_K. Save or upload the outputs before the free session expires.

In [ ]:
!bash scripts/build_llama_cpp.sh
!bash scripts/export_gguf.sh artifacts/fieldmind-merged-16bit artifacts/gguf
!bash scripts/quantize_tournament.sh artifacts/gguf/FieldMind-Africa-1.7B-F16.gguf artifacts/gguf
!ls -lh artifacts/gguf

Next: run `evaluate_gguf.py` and the official ADTC profiler for every candidate on the same four-thread CPU. Do not publish a winning quantization or performance number until the JSON reports exist. Hugging Face upload is intentionally left as an optional owner action so no token is required for training.